<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/LiquidityMgmt/generate_synthetic_liquidity_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthetic Liquidity Management Data Generator

This notebook generates realistic synthetic financial transaction data for liquidity management analysis using Monte Carlo simulations.

## Learning Objectives
- Understand liquidity management data structure
- Generate realistic synthetic financial data
- Explore transaction patterns and distributions
- Prepare data for Monte Carlo analysis

## Setup and Installation

In [ ]:
# Install required packages
!pip install faker pandas numpy matplotlib seaborn

# Import libraries
import csv
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
from datetime import datetime, timedelta
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Setup completed successfully!")

## Configuration Parameters

Adjust these parameters to customize your synthetic data generation:

In [ ]:
# Data generation parameters
NUM_ROWS = 10000          # Number of transactions to generate
LOCALE = 'en_IN'          # Locale for Indian context
AMOUNT_MEAN = 500000      # Mean transaction amount (INR)
AMOUNT_STDDEV = 200000    # Standard deviation for amounts
RANDOM_SEED = 42          # For reproducibility
OUTPUT_FILE = 'synthetic_liquidity_data.csv'

print(f"Configuration:")
print(f"📊 Rows to generate: {NUM_ROWS:,}")
print(f"🌍 Locale: {LOCALE}")
print(f"💰 Mean amount: ₹{AMOUNT_MEAN:,}")
print(f"📈 Std deviation: ₹{AMOUNT_STDDEV:,}")
print(f"🎲 Random seed: {RANDOM_SEED}")

## Data Generator Class

This class handles the generation of realistic synthetic liquidity data:

In [ ]:
class LiquidityDataGenerator:
    """
    A class to generate synthetic liquidity management data for financial analysis.
    """
    
    def __init__(self, locale: str = 'en_IN', seed: Optional[int] = None):
        self.locale = locale
        if seed is not None:
            random.seed(seed)
            Faker.seed(seed)
        
        try:
            self.fake = Faker(locale)
        except AttributeError:
            print(f"Warning: Locale {locale} not supported, using default")
            self.fake = Faker()
    
    def generate_transaction_type(self) -> str:
        """Generate transaction type with realistic distribution."""
        return random.choices(['inflow', 'outflow'], weights=[45, 55])[0]
    
    def generate_amount(self, transaction_type: str, mean: float, stddev: float) -> float:
        """Generate transaction amount based on type and parameters."""
        try:
            if transaction_type == 'inflow':
                adjusted_mean = mean * 1.2
            else:
                adjusted_mean = mean * 0.8
            
            amount = abs(random.gauss(adjusted_mean, stddev))
            amount = max(amount, 1000)
            
            return round(amount, 2)
        except Exception as e:
            print(f"Error generating amount: {e}")
            return 10000.0
    
    def generate_counterparty(self, transaction_type: str) -> str:
        """Generate realistic counterparty names."""
        try:
            if transaction_type == 'inflow':
                prefixes = ['', 'M/s ', 'Shri ', 'Smt ']
                return random.choice(prefixes) + self.fake.company()
            else:
                types = ['suppliers', 'vendors', 'employees']
                chosen_type = random.choice(types)
                
                if chosen_type == 'employees':
                    return self.fake.name()
                else:
                    return self.fake.company()
        except Exception as e:
            print(f"Error generating counterparty: {e}")
            return "Unknown Counterparty"
    
    def generate_description(self, transaction_type: str) -> str:
        """Generate realistic transaction descriptions."""
        try:
            if transaction_type == 'inflow':
                descriptions = [
                    "Revenue from operations", "Customer payment received",
                    "Investment income", "Interest income", "Sales proceeds",
                    "Service charges received", "Commission income"
                ]
            else:
                descriptions = [
                    "Supplier payment", "Salary disbursement", "Utility bills payment",
                    "Rent payment", "Equipment purchase", "Professional fees",
                    "Marketing expenses", "Office supplies", "Travel expenses"
                ]
            
            return random.choice(descriptions)
        except Exception as e:
            print(f"Error generating description: {e}")
            return "Financial transaction"
    
    def generate_data(self, num_rows: int, amount_mean: float, amount_stddev: float) -> pd.DataFrame:
        """Generate synthetic liquidity data and return as DataFrame."""
        try:
            print(f"Generating {num_rows} synthetic liquidity management records...")
            
            data = []
            
            for i in range(1, num_rows + 1):
                transaction_id = f'TXN{i:08d}'
                transaction_date = self.fake.date_between(start_date='-2y', end_date='today')
                transaction_type = self.generate_transaction_type()
                counterparty = self.generate_counterparty(transaction_type)
                amount = self.generate_amount(transaction_type, amount_mean, amount_stddev)
                description = self.generate_description(transaction_type)
                currency = 'INR'
                business_unit = random.choice(['Corporate Banking', 'Retail Banking', 'Investment Banking', 'Treasury'])
                account_type = random.choice(['Current Account', 'Savings Account', 'Fixed Deposit', 'Call Money'])
                
                data.append([
                    transaction_id, transaction_date, transaction_type,
                    counterparty, amount, description, currency,
                    business_unit, account_type
                ])
                
                if i % 1000 == 0:
                    print(f"Generated {i} records...")
            
            columns = [
                'transaction_id', 'transaction_date', 'transaction_type',
                'counterparty', 'amount_inr', 'description', 'currency',
                'business_unit', 'account_type'
            ]
            
            df = pd.DataFrame(data, columns=columns)
            df['transaction_date'] = pd.to_datetime(df['transaction_date'])
            
            print(f"✅ Successfully generated {len(df)} synthetic records")
            return df
            
        except Exception as e:
            print(f"❌ Error generating synthetic data: {e}")
            return pd.DataFrame()

print("✅ Data generator class defined successfully!")

## Generate Synthetic Data

Let's generate the synthetic liquidity data:

In [ ]:
# Initialize generator
generator = LiquidityDataGenerator(locale=LOCALE, seed=RANDOM_SEED)

# Generate data
df = generator.generate_data(
    num_rows=NUM_ROWS,
    amount_mean=AMOUNT_MEAN,
    amount_stddev=AMOUNT_STDDEV
)

# Display basic information
print(f"\n📊 Dataset Summary:")
print(f"Shape: {df.shape}")
print(f"Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")
print(f"Total amount: ₹{df['amount_inr'].sum():,.2f}")

# Display first few rows
print(f"\n🔍 First 5 rows:")
df.head()

## Data Analysis and Visualization

Let's analyze the generated data to understand its characteristics:

In [ ]:
# Basic statistics
print("📈 Transaction Type Distribution:")
print(df['transaction_type'].value_counts())
print(f"\nInflow ratio: {(df['transaction_type'] == 'inflow').mean():.2%}")
print(f"Outflow ratio: {(df['transaction_type'] == 'outflow').mean():.2%}")

print("\n💰 Amount Statistics:")
print(df['amount_inr'].describe())

print("\n🏢 Business Unit Distribution:")
print(df['business_unit'].value_counts())

print("\n🏦 Account Type Distribution:")
print(df['account_type'].value_counts())

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Transaction type distribution
df['transaction_type'].value_counts().plot(kind='pie', ax=axes[0,0], autopct='%1.1f%%')
axes[0,0].set_title('Transaction Type Distribution')
axes[0,0].set_ylabel('')

# 2. Amount distribution by transaction type
df.boxplot(column='amount_inr', by='transaction_type', ax=axes[0,1])
axes[0,1].set_title('Amount Distribution by Transaction Type')
axes[0,1].set_xlabel('Transaction Type')
axes[0,1].set_ylabel('Amount (INR)')

# 3. Daily transaction count over time
daily_counts = df.groupby(df['transaction_date'].dt.date).size()
daily_counts.plot(ax=axes[0,2], alpha=0.7)
axes[0,2].set_title('Daily Transaction Count Over Time')
axes[0,2].set_xlabel('Date')
axes[0,2].set_ylabel('Number of Transactions')
axes[0,2].tick_params(axis='x', rotation=45)

# 4. Business unit distribution
df['business_unit'].value_counts().plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Business Unit Distribution')
axes[1,0].set_xlabel('Business Unit')
axes[1,0].set_ylabel('Count')
axes[1,0].tick_params(axis='x', rotation=45)

# 5. Amount histogram
axes[1,1].hist(df['amount_inr'], bins=50, alpha=0.7, edgecolor='black')
axes[1,1].set_title('Amount Distribution')
axes[1,1].set_xlabel('Amount (INR)')
axes[1,1].set_ylabel('Frequency')

# 6. Monthly transaction volume
df['month'] = df['transaction_date'].dt.to_period('M')
monthly_volume = df.groupby('month')['amount_inr'].sum()
monthly_volume.plot(kind='bar', ax=axes[1,2])
axes[1,2].set_title('Monthly Transaction Volume')
axes[1,2].set_xlabel('Month')
axes[1,2].set_ylabel('Total Amount (INR)')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("📊 Visualizations completed!")

## Save Data

Save the generated data to CSV for use in Monte Carlo simulation:

In [ ]:
# Save to CSV
df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Data saved to '{OUTPUT_FILE}'")
print(f"📁 File size: {len(df)} rows, {df.shape[1]} columns")

# Display file info
import os
file_size = os.path.getsize(OUTPUT_FILE)
print(f"💾 File size on disk: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")

# Verify saved data
print("\n🔍 Verification - reading saved file:")
df_verify = pd.read_csv(OUTPUT_FILE)
print(f"Loaded {len(df_verify)} rows from saved file")
print("✅ Data saved and verified successfully!")

## Self-Service Exploration

### For Students:

Try modifying the parameters above to explore different scenarios:

1. **Change data size**: Modify `NUM_ROWS` to generate different dataset sizes
2. **Adjust distributions**: Change `AMOUNT_MEAN` and `AMOUNT_STDDEV` to see different transaction patterns
3. **Experiment with ratios**: Modify the weights in `generate_transaction_type()` to change inflow/outflow ratios
4. **Date ranges**: Adjust the date range in `generate_data()` method

### For Banking Applications:

This synthetic data can be used to:

1. **Test liquidity models** without exposing real customer data
2. **Train staff** on liquidity management concepts
3. **Validate Monte Carlo simulations** before production deployment
4. **Stress test** liquidity management systems
5. **Regulatory reporting** practice and validation

### Next Steps:

1. Use this data with the Monte Carlo simulation notebook
2. Experiment with different statistical distributions
3. Add seasonal patterns to make data more realistic
4. Incorporate correlation between different transaction types